In [ ]:
# --- Interactive PSF Extractor (Looping + Rotation + Side View Toggle) ---
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.widgets import RectangleSelector
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
import numpy as np
import tifffile
from pathlib import Path
import opym

# --- 1. UI Elements ---
print("1. Select the DSR (Deskewed) folder containing your beads:")
dsr_chooser = FileChooser(
    path="/mmfs2/scratch/SDSMT.LOCAL/bscott/DataUpload",
    title="<b>Select DSR Folder:</b>",
    show_only_dirs=True
)

# Sliders
channel_slider = widgets.IntSlider(description="Channel:", min=0, max=0, disabled=True)
slider_max_input = widgets.FloatText(value=255.0, description="Max Range:", layout=widgets.Layout(width='150px'), disabled=True)
contrast_slider = widgets.FloatRangeSlider(description="Contrast:", min=0, max=255, step=1, value=[0, 255], layout=widgets.Layout(width='400px'), disabled=True)

# Buttons
btn_load = widgets.Button(description="Load Data", button_style='primary')
btn_rotate = widgets.Button(description="Rotate XY 90°", button_style='info', disabled=True, icon='repeat')
btn_check = widgets.Button(description="Verify Crop", button_style='info', disabled=True, icon='eye') 

# Step 2 Controls
btn_step2 = widgets.Button(description="Confirm XY -> Go to Z", button_style='warning', disabled=True)
btn_toggle_side = widgets.Button(description="Side: XZ (Switch to YZ)", button_style='info', disabled=True, icon='exchange')

btn_save = widgets.Button(description="Save & Pick Next", button_style='success', disabled=True)

# Output Areas
log_output = widgets.Output(layout={'border': '1px solid #ccc', 'height': '150px', 'overflow_y': 'scroll'})
plot_output = widgets.Output()

# Application State
state = {
    "stack": None,       
    "get_stack_fn": None,
    "t_idx": 0,          
    "c_idx": 0,          
    "xy_roi": None,      
    "z_roi": None,       
    "selectors": [],     
    "fig": None,
    "ax": None,
    "img_obj": None,     
    "dsr_path": None,
    "step": 1,
    "done_rois": [],      
    "rot": 0,             # XY Rotation (0-3)
    "side_mode": "XZ"     # "XZ" or "YZ"
}

# --- 2. Coordinate Transforms ---
def transform_point_inverse(sx, sy, shape, k):
    h, w = shape
    k = k % 4
    if k == 0: return sx, sy
    if k == 1: return w - 1 - sy, sx
    if k == 2: return w - 1 - sx, h - 1 - sy
    if k == 3: return sy, h - 1 - sx
    return sx, sy

def transform_point_forward(dx, dy, shape, k):
    h, w = shape
    k = k % 4
    if k == 0: return dx, dy
    if k == 1: return dy, w - 1 - dx
    if k == 2: return w - 1 - dx, h - 1 - dy
    if k == 3: return h - 1 - dy, dx
    return dx, dy

def get_rotated_dims(shape, k):
    h, w = shape
    if k % 2 == 1: return w, h
    return h, w

# --- 3. Event Handlers ---

def update_display():
    if state["stack"] is None or state["img_obj"] is None: return
    vmin, vmax = contrast_slider.value
    state["img_obj"].set_clim(vmin, vmax)
    state["fig"].canvas.draw_idle()

def on_rotate_click(b):
    if state["step"] != 1: return
    state["rot"] = (state["rot"] + 1) % 4
    start_step_1_xy()

def on_toggle_side_click(b):
    """Switches between XZ and YZ views in Step 2."""
    if state["step"] != 2: return
    
    # Toggle State
    if state["side_mode"] == "XZ":
        state["side_mode"] = "YZ"
        btn_toggle_side.description = "Side: YZ (Switch to XZ)"
    else:
        state["side_mode"] = "XZ"
        btn_toggle_side.description = "Side: XZ (Switch to YZ)"
        
    # Re-draw Step 2
    start_step_2_xz(None)

def on_channel_change(change):
    if change['type'] != 'change' or change['name'] != 'value': return
    state["c_idx"] = change['new']
    with log_output:
        print(f"Loading Channel {state['c_idx']}...")
        try:
            state["stack"] = state["get_stack_fn"](state["t_idx"], state["c_idx"])
            if state["step"] == 1: start_step_1_xy()
            elif state["step"] == 2: start_step_2_xz(None)
        except Exception as e: print(f"Error: {e}")

def on_load_click(b):
    with log_output:
        clear_output()
        if not dsr_chooser.selected: return
        path = Path(dsr_chooser.selected)
        state["dsr_path"] = path
        state["done_rois"] = [] 
        state["rot"] = 0 
        print(f"Reading {path.name}...")
        try:
            (get_stack, t_min, t_max, c_min, c_max, z_max, y_dim, x_dim, _) = opym.load_tiff_series(path)
            state["get_stack_fn"] = get_stack
            state["t_idx"] = t_min
            state["c_idx"] = c_min
            state["stack"] = get_stack(t_min, c_min)
            
            # UI Updates
            channel_slider.min, channel_slider.max = c_min, c_max
            channel_slider.value = c_min
            channel_slider.disabled = False
            
            d_max = np.max(state["stack"])
            lim = 65535.0 if d_max > 255 else 255.0
            slider_max_input.value = lim
            slider_max_input.disabled = False
            contrast_slider.max = lim
            
            p1, p99 = np.percentile(state["stack"], [1, 99.9])
            if p99 > lim: p99 = lim
            contrast_slider.value = (float(p1), float(p99))
            contrast_slider.disabled = False
            
            btn_rotate.disabled = False
            btn_step2.disabled = False
            btn_toggle_side.disabled = True # Only active in Step 2
            
            print(f"✅ Loaded. Volume Shape: {state['stack'].shape}")
            start_step_1_xy()
        except Exception as e: 
            print(f"Load Error: {e}")
            import traceback
            traceback.print_exc()

def on_check_click(b):
    if state["xy_roi"] is None or state["step"] != 1: return
    x1, x2, y1, y2 = state["xy_roi"]
    h, w = state["stack"].shape[1], state["stack"].shape[2]
    sx1, sy1 = transform_point_forward(x1, y1, (h, w), state["rot"])
    sx2, sy2 = transform_point_forward(x2, y2, (h, w), state["rot"])
    rx, ry = min(sx1, sx2), min(sy1, sy2)
    rw, rh = abs(sx2 - sx1), abs(sy2 - sy1)
    rect = patches.Rectangle((rx, ry), rw, rh, linewidth=2, edgecolor='yellow', facecolor='none')
    state["ax"].add_patch(rect)
    state["fig"].canvas.draw_idle()

# --- 4. Plotting & Selection ---

def on_select_xy(eclick, erelease):
    sx1, sx2 = sorted([int(eclick.xdata), int(erelease.xdata)])
    sy1, sy2 = sorted([int(eclick.ydata), int(erelease.ydata)])
    h_data, w_data = state["stack"].shape[1], state["stack"].shape[2]
    x1a, y1a = transform_point_inverse(sx1, sy1, (h_data, w_data), state["rot"])
    x2a, y2a = transform_point_inverse(sx2, sy2, (h_data, w_data), state["rot"])
    x_final = sorted([x1a, x2a])
    y_final = sorted([y1a, y2a])
    state["xy_roi"] = (x_final[0], x_final[1], y_final[0], y_final[1])
    btn_check.disabled = False
    with log_output:
        print(f"Selected: x[{x_final[0]}:{x_final[1]}], y[{y_final[0]}:{y_final[1]}]")

def on_select_z(eclick, erelease):
    # In both XZ and YZ views, the Y-axis of the plot is Z-slices
    z1, z2 = sorted([int(eclick.ydata), int(erelease.ydata)])
    state["z_roi"] = (z1, z2)
    btn_save.disabled = False
    # Optional: Log which view made the selection
    # with log_output: print(f"Z Selected ({state['side_mode']}): z[{z1}:{z2}]")

def start_step_1_xy():
    state["step"] = 1
    state["z_roi"] = None
    btn_check.disabled = True
    btn_toggle_side.disabled = True # Disable side toggle in XY view
    
    with plot_output:
        clear_output(wait=True)
        mip_xy = np.max(state["stack"], axis=0)
        mip_disp = np.rot90(mip_xy, k=state["rot"])
        
        fig, ax = plt.subplots(figsize=(8, 8))
        vmin, vmax = contrast_slider.value
        img = ax.imshow(mip_disp, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')
        ax.set_title(f"Step 1: Select Bead (Rot: {state['rot']*90}°)")
        
        # Draw Previous ROIs
        h_data, w_data = mip_xy.shape
        for roi in state["done_rois"]:
            dx1, dx2, dy1, dy2 = roi
            sx1, sy1 = transform_point_forward(dx1, dy1, (h_data, w_data), state["rot"])
            sx2, sy2 = transform_point_forward(dx2, dy2, (h_data, w_data), state["rot"])
            rx, ry = min(sx1, sx2), min(sy1, sy2)
            rw, rh = abs(sx2 - sx1), abs(sy2 - sy1)
            rect = patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='red', facecolor='none', linestyle='--')
            ax.add_patch(rect)
        
        state["fig"] = fig
        state["ax"] = ax
        state["img_obj"] = img
        rs = RectangleSelector(ax, on_select_xy, useblit=True, button=[1], interactive=True)
        state["selectors"] = [rs]
        plt.show()

def start_step_2_xz(b):
    if state["xy_roi"] is None:
        with log_output: print("⚠️ Draw a box on XY first!")
        return

    state["step"] = 2
    btn_toggle_side.disabled = False # Enable toggle
    x1, x2, y1, y2 = state["xy_roi"]
    
    try:
        crop = state["stack"][:, y1:y2, x1:x2]
        if crop.size == 0:
            with log_output: print("❌ Error: Crop is 0 pixels! Check selection.")
            return

        # --- SMART TOGGLE LOGIC ---
        if state["side_mode"] == "XZ":
            # Project along Y (axis 1) -> Result (Z, X)
            mip_side = np.max(crop, axis=1)
            x_label = "X (Cropped)"
        else:
            # Project along X (axis 2) -> Result (Z, Y)
            mip_side = np.max(crop, axis=2)
            x_label = "Y (Cropped)"
        # --------------------------

        with plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(8, 5))
            vmin, vmax = contrast_slider.value
            img = ax.imshow(mip_side, cmap='gray', vmin=vmin, vmax=vmax, aspect='auto', origin='upper')
            
            ax.set_title(f"Step 2: Crop Z Range ({state['side_mode']} View)")
            ax.set_xlabel(x_label)
            ax.set_ylabel("Z Slices")
            
            state["fig"] = fig
            state["ax"] = ax
            state["img_obj"] = img
            rs = RectangleSelector(ax, on_select_z, useblit=True, button=[1], interactive=True)
            state["selectors"] = [rs]
            plt.show()
    except Exception as e:
        with log_output: print(f"Error in Step 2: {e}")

def on_save_click(b):
    if state["z_roi"] is None:
        with log_output: print("⚠️ Select Z range first.")
        return
    z1, z2 = state["z_roi"]
    x1, x2, y1, y2 = state["xy_roi"]
    final_vol = state["stack"][z1:z2, y1:y2, x1:x2]
    
    # Auto-Name
    base = f"PSF_C{state['c_idx']}_Bead"
    ctr = 1
    while True:
        fname = f"{base}_{ctr:03d}.tif"
        if not (state["dsr_path"] / fname).exists(): break
        ctr += 1
    
    tifffile.imwrite(state["dsr_path"] / fname, final_vol)
    with log_output: print(f"✅ Saved #{ctr}: {fname}")
    state["done_rois"].append((x1, x2, y1, y2))
    btn_save.disabled = True
    start_step_1_xy()

# --- 5. Wire Up ---
channel_slider.observe(on_channel_change, names='value')
slider_max_input.observe(lambda c: setattr(contrast_slider, 'max', c['new']) if c['type']=='change' and c['name']=='value' else None, names='value')
contrast_slider.observe(lambda c: update_display(), names='value') 

btn_load.on_click(on_load_click)
btn_rotate.on_click(on_rotate_click)
btn_check.on_click(on_check_click)
btn_step2.on_click(start_step_2_xz)
btn_toggle_side.on_click(on_toggle_side_click)
btn_save.on_click(on_save_click)

ui = widgets.VBox([
    dsr_chooser,
    widgets.HBox([channel_slider, slider_max_input, contrast_slider]),
    widgets.HBox([btn_load, btn_rotate, btn_check]),
    widgets.HBox([btn_step2, btn_toggle_side, btn_save]),
    log_output,
    plot_output
])
display(ui)

In [ ]:
# --- Verify Saved PSF (Orthogonal Views) ---
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
import tifffile
import numpy as np
from pathlib import Path

# --- UI Setup ---
print("Select the Saved PSF File to Verify:")
psf_chooser = FileChooser(
    path="/mmfs2/scratch/SDSMT.LOCAL/bscott/DataUpload",
    title="<b>Select PSF TIFF:</b>",
    filter_pattern=['*.tif', '*.tiff']
)

btn_verify = widgets.Button(description="Load & Inspect", button_style='info')
verify_output = widgets.Output()

def plot_orthogonal(vol, title):
    """Generates a 3-panel orthogonal view (XY, XZ, YZ)"""
    # Projections
    mip_xy = np.max(vol, axis=0) # Top-down
    mip_xz = np.max(vol, axis=1) # Side (Front)
    mip_yz = np.max(vol, axis=2) # Side (Left)
    
    # Setup Figure
    fig = plt.figure(figsize=(9, 5))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1])
    
    # 1. XY View (Top Left)
    ax_xy = fig.add_subplot(gs[0, 0])
    ax_xy.imshow(mip_xy, cmap='gray', origin='upper')
    ax_xy.set_title(f"XY (Top) {mip_xy.shape}")
    ax_xy.set_ylabel("Y")
    ax_xy.set_xlabel("X")
    
    # 2. XZ View (Bottom Left) - shares X with XY
    ax_xz = fig.add_subplot(gs[1, 0], sharex=ax_xy)
    ax_xz.imshow(mip_xz, cmap='gray', origin='upper', aspect='auto')
    ax_xz.set_title(f"XZ (Side) {mip_xz.shape}")
    ax_xz.set_ylabel("Z")
    
    # 3. YZ View (Top Right) - shares Y with XY
    ax_yz = fig.add_subplot(gs[0, 1], sharey=ax_xy)
    # Transpose YZ to match Y-axis orientation of XY plot
    ax_yz.imshow(mip_yz.T, cmap='gray', origin='upper', aspect='auto')
    ax_yz.set_title(f"YZ (Side) {mip_yz.shape}")
    ax_yz.set_xlabel("Z")
    
    # Metadata Text (Bottom Right)
    ax_text = fig.add_subplot(gs[1, 1])
    ax_text.axis('off')
    info = (
        f"Filename: {title}\n"
        f"Shape: {vol.shape} (Z, Y, X)\n"
        f"DType: {vol.dtype}\n"
        f"Min/Max: {vol.min()} / {vol.max()}\n"
    )
    ax_text.text(0.1, 0.5, info, fontsize=10, va='center')
    
    plt.tight_layout()
    plt.show()

def on_verify_click(b):
    with verify_output:
        clear_output()
        if not psf_chooser.selected:
            print("❌ Please select a file.")
            return
            
        fpath = Path(psf_chooser.selected)
        print(f"Loading: {fpath.name}...")
        
        try:
            # Load Tiff directly
            vol = tifffile.imread(fpath)
            
            # Check dimensions
            if vol.ndim != 3:
                print(f"⚠️ Warning: Expected 3D (Z,Y,X), got {vol.ndim}D shape {vol.shape}.")
                # Handle 4D if user accidentally saved time/channel dims
                if vol.ndim == 4: 
                    print("Attempting to squeeze...")
                    vol = np.squeeze(vol)
            
            plot_orthogonal(vol, fpath.name)
            
        except Exception as e:
            print(f"❌ Error: {e}")

btn_verify.on_click(on_verify_click)

display(psf_chooser)
display(btn_verify)
display(verify_output)

In [ ]:
# --- Bead Inspector v4.1 (High Precision + Saves Masks) ---
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path as MplPath
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
import tifffile
import numpy as np
from pathlib import Path
from scipy import ndimage
from skimage.filters import threshold_otsu
import shutil

print("Select Folder containing your Bead ROIs:")
folder_chooser = FileChooser(
    path="/mmfs2/scratch/SDSMT.LOCAL/bscott/DataUpload",
    title="<b>Select Folder:</b>",
    show_only_dirs=True
)

# --- State ---
state = {
    "files": [],
    "current_idx": 0,
    "current_vol": None,
    "current_mask": None, 
    "kept": [],
    "rejected": [],
    "fig": None,
    "ax_img": None,
    "ax_prof": None,
    "poly": None,         
    "mode": "Auto"        
}

# --- UI Controls ---
# UPDATED: Finer step size (0.01) and wider range
sens_slider = widgets.FloatSlider(
    value=1.00, min=0.10, max=5.00, step=0.01,
    description="Sensitivity:",
    continuous_update=False,
    readout_format='.2f',
    layout=widgets.Layout(width='300px')
)

btn_start = widgets.Button(description="Start Inspection", button_style='primary')
btn_draw = widgets.Button(description="✏️ Draw Manual ROI", button_style='info', icon='pencil')
btn_keep = widgets.Button(description="✅ KEEP & SAVE MASK", button_style='success', layout=widgets.Layout(width='180px'))
btn_reject = widgets.Button(description="❌ REJECT", button_style='danger', layout=widgets.Layout(width='100px'))
btn_undo = widgets.Button(description="Undo", button_style='warning', disabled=True)
btn_finish = widgets.Button(description="Finish & Move Rejected", button_style='info', disabled=True)

progress_label = widgets.Label(value="Waiting to start...")
plot_output = widgets.Output()

# --- Logic ---

def get_auto_mask(vol, sensitivity=1.0):
    smooth = ndimage.gaussian_filter(vol, sigma=1.0)
    try: thresh = threshold_otsu(smooth)
    except: thresh = np.percentile(smooth, 90)
    
    # Higher Sensitivity = Lower Threshold (Detects more faint signal)
    eff_thresh = thresh / sensitivity
    
    mask = smooth > eff_thresh
    mask = ndimage.binary_dilation(mask, iterations=2)
    return mask

def calculate_bg(vol, mask):
    bg_pixels = vol[~mask]
    if len(bg_pixels) == 0: return np.percentile(vol, 5)
    return np.median(bg_pixels)

def on_poly_select(verts):
    if state["current_vol"] is None: return
    vol = state["current_vol"]
    h, w = vol.shape[1], vol.shape[2] 
    y, x = np.mgrid[:h, :w]
    points = np.transpose((x.ravel(), y.ravel()))
    path = MplPath(verts)
    mask_flat = path.contains_points(points)
    mask_2d = mask_flat.reshape((h, w))
    mask_3d = np.broadcast_to(mask_2d[None, :, :], vol.shape)
    
    state["current_mask"] = mask_3d
    state["mode"] = "Manual"
    update_plots()
    btn_draw.description = "✏️ Redraw ROI"
    btn_draw.button_style = 'info'

def toggle_draw_mode(b):
    if state["poly"] is None: return
    btn_draw.description = "Click points..."
    btn_draw.button_style = 'warning'
    state["poly"].disconnect_events()
    state["poly"] = PolygonSelector(state["ax_img"], on_poly_select, useblit=True)

def on_sensitivity_change(change):
    if state["mode"] == "Manual": return 
    if state["current_vol"] is None: return
    state["current_mask"] = get_auto_mask(state["current_vol"], sens_slider.value)
    update_plots()

def load_bead(idx):
    if idx >= len(state["files"]):
        with plot_output:
            clear_output()
            print("🎉 Inspection Complete!")
            print(f"Kept: {len(state['kept'])}")
            print(f"Rejected: {len(state['rejected'])}")
        btn_keep.disabled = True
        btn_reject.disabled = True
        btn_finish.disabled = False
        return

    f = state["files"][idx]
    vol = tifffile.imread(f).astype(np.float32)
    if vol.ndim > 3: vol = np.squeeze(vol)
    
    state["current_vol"] = vol
    state["mode"] = "Auto"
    state["current_mask"] = get_auto_mask(vol, sens_slider.value)
    
    progress_label.value = f"File {idx+1}/{len(state['files'])}: {f.name}"
    
    if state["fig"] is None:
        with plot_output:
            clear_output(wait=True)
            fig = plt.figure(figsize=(9, 5))
            ax_img = fig.add_subplot(121)
            ax_prof = fig.add_subplot(122)
            state["fig"] = fig
            state["ax_img"] = ax_img
            state["ax_prof"] = ax_prof
            state["poly"] = PolygonSelector(ax_img, on_poly_select, useblit=True)
            plt.show()
    
    update_plots()

def update_plots():
    vol = state["current_vol"]
    mask = state["current_mask"]
    bg_val = calculate_bg(vol, mask)
    mid_y = vol.shape[1] // 2
    img_slice = vol[:, mid_y, :]
    mask_slice = mask[:, mid_y, :]
    
    ax1 = state["ax_img"]
    ax1.clear()
    ax1.imshow(img_slice, cmap='gray', origin='upper', aspect='auto')
    if np.any(mask_slice):
        ax1.contour(mask_slice, levels=[0.5], colors='red', linewidths=1) # Thinner line for precision
    
    mode_txt = f"Mode: {state['mode']}"
    if state["mode"] == "Auto": mode_txt += f" (Sens: {sens_slider.value:.2f})"
    ax1.set_title(f"Bead Detection (XZ) - {mode_txt}")
    ax1.axis('off')
    
    state["poly"] = PolygonSelector(ax1, on_poly_select, useblit=True)
    
    ax2 = state["ax_prof"]
    ax2.clear()
    mid_x = vol.shape[2] // 2
    profile = vol[:, mid_y, mid_x]
    ax2.plot(profile, label='Signal', color='blue', alpha=0.7)
    ax2.axhline(bg_val, color='red', linestyle='--', linewidth=1.5, label=f'BG: {bg_val:.1f}')
    ax2.set_title("Z-Profile (Center)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    state["fig"].canvas.draw_idle()

# --- Saving Logic ---
def on_keep(b):
    current_file = state["files"][state["current_idx"]]
    mask_name = current_file.parent / (current_file.stem + "_mask.tif")
    mask_uint8 = (state["current_mask"] * 255).astype(np.uint8)
    tifffile.imwrite(mask_name, mask_uint8)
    
    state["kept"].append(current_file)
    state["current_idx"] += 1
    btn_undo.disabled = False
    load_bead(state["current_idx"])

def on_reject(b):
    state["rejected"].append(state["files"][state["current_idx"]])
    state["current_idx"] += 1
    btn_undo.disabled = False
    load_bead(state["current_idx"])

def on_undo(b):
    if state["current_idx"] > 0:
        state["current_idx"] -= 1
        f = state["files"][state["current_idx"]]
        if f in state["kept"]: 
            state["kept"].remove(f)
            mask_name = f.parent / (f.stem + "_mask.tif")
            if mask_name.exists(): mask_name.unlink()
        if f in state["rejected"]: state["rejected"].remove(f)
        load_bead(state["current_idx"])

def on_finish(b):
    if not state["rejected"]: return
    rej_dir = Path(folder_chooser.selected) / "rejected_beads"
    rej_dir.mkdir(exist_ok=True)
    for f in state["rejected"]:
        try: shutil.move(str(f), str(rej_dir / f.name))
        except: pass
    print(f"Moved {len(state['rejected'])} files to {rej_dir.name}")

# --- Wiring ---
btn_start.on_click(on_start)
btn_draw.on_click(toggle_draw_mode)
sens_slider.observe(on_sensitivity_change, names='value')
btn_keep.on_click(on_keep)
btn_reject.on_click(on_reject)
btn_undo.on_click(on_undo)
btn_finish.on_click(on_finish)

controls = widgets.VBox([
    widgets.HBox([sens_slider, btn_draw]),
    widgets.HBox([btn_keep, btn_reject, btn_undo])
])

display(widgets.VBox([
    folder_chooser,
    btn_start,
    widgets.HBox([plot_output]),
    progress_label,
    controls,
    widgets.HTML("<hr>"),
    btn_finish
]))

In [ ]:
# --- Averager v4 (Strict "Cookie Cutter" Mode) ---
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
import tifffile
import numpy as np
from pathlib import Path
from scipy import ndimage, signal, optimize
from skimage.filters import threshold_otsu

print("Select Folder containing your KEPT Bead ROIs:")
folder_chooser = FileChooser(
    path="/mmfs2/scratch/SDSMT.LOCAL/bscott/DataUpload",
    title="<b>Select Folder:</b>",
    show_only_dirs=True
)

# New Control: Strict Masking
strict_mask_check = widgets.Checkbox(value=True, description="Hard Crop (Zero outside mask)")
taper_slider = widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05, description="Edge Taper:")

btn_run = widgets.Button(description="Generate Master PSF", button_style='success', layout=widgets.Layout(width='200px'))
out_log = widgets.Output(layout={'border': '1px solid #ccc', 'height': '200px', 'overflow_y': 'scroll'})
out_plot = widgets.Output()

def fit_gaussian_1d(x, y):
    def gauss(x, a, x0, sigma, offset):
        return a * np.exp(-(x - x0)**2 / (2 * sigma**2)) + offset
    try:
        p0 = [np.max(y), len(y)/2, len(y)/4, 0]
        popt, _ = optimize.curve_fit(gauss, x, y, p0=p0)
        return 2.355 * abs(popt[2]) 
    except:
        return 0.0

def get_fallback_mask(vol):
    smooth = ndimage.gaussian_filter(vol, sigma=1.0)
    try: thresh = threshold_otsu(smooth)
    except: thresh = np.percentile(smooth, 90)
    mask = smooth > thresh
    return ndimage.binary_dilation(mask, iterations=2)

def taper_vol(vol, fraction=0.1):
    d, h, w = vol.shape
    win_z = signal.windows.tukey(d, alpha=fraction)
    win_y = signal.windows.tukey(h, alpha=fraction)
    win_x = signal.windows.tukey(w, alpha=fraction)
    return vol * (win_z[:,None,None] * win_y[None,:,None] * win_x[None,None,:])

def run_average(b):
    with out_log:
        clear_output()
        if not folder_chooser.selected: return
        p = Path(folder_chooser.selected)
        
        all_tifs = sorted(list(p.glob("*.tif")) + list(p.glob("*.tiff")))
        bead_files = [f for f in all_tifs if "Master" not in f.name and "Avg" not in f.name and "_mask" not in f.name]
        
        if not bead_files:
            print("❌ No beads found.")
            return
            
        print(f"--- Averaging {len(bead_files)} Beads ---")
        
        # Determine Canvas
        shapes = [tifffile.imread(f).shape for f in bead_files]
        mz = max((s[0] if len(s)==3 else 1) for s in shapes)
        my = max((s[1] if len(s)==3 else s[0]) for s in shapes)
        mx = max((s[2] if len(s)==3 else s[1]) for s in shapes)
        mz += 1 if mz%2==0 else 0
        my += 1 if my%2==0 else 0
        mx += 1 if mx%2==0 else 0
        target_shape = (mz, my, mx)
        center_target = np.array(target_shape) / 2.0
        
        aligned_sum = np.zeros(target_shape, dtype=np.float32)
        used_count = 0
        
        for f in bead_files:
            vol = tifffile.imread(f).astype(np.float32)
            if vol.ndim > 3: vol = np.squeeze(vol)
            
            # 1. LOAD MASK
            mask_path = f.parent / (f.stem + "_mask.tif")
            if mask_path.exists():
                mask = tifffile.imread(mask_path) > 0
                mask_source = "USER"
            else:
                mask = get_fallback_mask(vol)
                mask_source = "AUTO"
            
            # 2. Subtract BG
            bg_pixels = vol[~mask]
            bg_val = np.median(bg_pixels) if len(bg_pixels) > 0 else 0
            vol_clean = vol - bg_val
            vol_clean[vol_clean < 0] = 0
            
            # 3. STRICT MASKING (The Cookie Cutter)
            # If enabled, force everything outside the mask to exactly 0
            if strict_mask_check.value:
                vol_clean[~mask] = 0.0
            
            # 4. Center of Mass (Weighted by Clean Signal)
            if np.sum(vol_clean) == 0:
                print(f"⚠️ Skipped {f.name} (Empty after cleaning)")
                continue
                
            com = np.array(ndimage.center_of_mass(vol_clean))
            
            # 5. Embed & Shift
            canvas = np.zeros(target_shape, dtype=np.float32)
            z_off = (mz - vol.shape[0]) // 2
            y_off = (my - vol.shape[1]) // 2
            x_off = (mx - vol.shape[2]) // 2
            canvas[z_off:z_off+vol.shape[0], y_off:y_off+vol.shape[1], x_off:x_off+vol.shape[2]] = vol_clean
            
            com_canvas = com + np.array([z_off, y_off, x_off])
            shift_vec = center_target - com_canvas
            
            aligned = ndimage.shift(canvas, shift_vec, order=3, prefilter=True)
            aligned[aligned < 0] = 0
            
            # Normalize Peak
            if np.max(aligned) > 0: aligned /= np.max(aligned)
            aligned_sum += aligned
            used_count += 1
            
            print(f"[{mask_source}] Merged: {f.name}")

        if used_count == 0: return

        # Final Average
        master_psf = aligned_sum / used_count
        
        # Taper
        if taper_slider.value > 0:
            master_psf = taper_vol(master_psf, fraction=taper_slider.value)
            
        master_psf /= np.max(master_psf)
        save_path = p / "Master_PSF_Final_Strict.tif"
        tifffile.imwrite(save_path, master_psf.astype(np.float32))
        print(f"\n✅ SAVED: {save_path.name}")
        
        # --- Visualization ---
        with out_plot:
            clear_output()
            
            cz, cy, cx = list(map(int, center_target))
            prof_z = master_psf[:, cy, cx]
            prof_x = master_psf[cz, cy, :]
            
            px_xy, px_z = 0.136, 0.100 
            fz = fit_gaussian_1d(np.arange(len(prof_z)), prof_z) * px_z
            fx = fit_gaussian_1d(np.arange(len(prof_x)), prof_x) * px_xy
            
            fig = plt.figure(figsize=(10, 6))
            gs = fig.add_gridspec(2, 3)
            
            # Log Scale helps see if background is truly gone
            ax0 = fig.add_subplot(gs[0, 0])
            ax0.imshow(np.log1p(master_psf[:, cy, :]), cmap='magma', aspect='auto')
            ax0.set_title(f"XZ View (Log Scale)\nFWHM Z: {fz:.3f} µm")
            
            ax1 = fig.add_subplot(gs[0, 1])
            ax1.imshow(np.log1p(master_psf[cz, :, :]), cmap='magma')
            ax1.set_title(f"XY View (Log Scale)\nFWHM X: {fx:.3f} µm")
            
            ax3 = fig.add_subplot(gs[1, :])
            ax3.plot(prof_z, label='Z Profile', color='blue')
            ax3.plot(prof_x, label='X Profile', color='orange', linestyle='--')
            ax3.set_title("Intensity Profiles")
            ax3.legend()
            ax3.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()

btn_run.on_click(run_average)
display(widgets.VBox([folder_chooser, widgets.HBox([strict_mask_check, taper_slider]), btn_run, out_log, out_plot]))

In [ ]:
# --- Bead Inspector v9 (Local Thresholding within Cylinder) ---
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
import tifffile
import numpy as np
from pathlib import Path
from scipy import ndimage
from skimage.filters import threshold_otsu
import shutil

print("Select Folder containing your Bead ROIs:")
folder_chooser = FileChooser(
    path="/mmfs2/scratch/SDSMT.LOCAL/bscott/DataUpload",
    title="<b>Select Folder:</b>",
    show_only_dirs=True
)

# --- State ---
state = {
    "files": [],
    "current_idx": 0,
    "current_vol": None,
    "current_mask": None, 
    "centroid": None, # (y, x)
    "kept": [],
    "rejected": [],
    "fig": None,
    "axes": {},
    "cid": None
}

# --- UI Controls ---
radius_slider = widgets.IntSlider(
    value=30, min=5, max=200, step=1,
    description="1. XY Radius (px):",
    continuous_update=True,
    layout=widgets.Layout(width='300px')
)

sens_slider = widgets.FloatSlider(
    value=1.00, min=0.10, max=5.00, step=0.05,
    description="2. Sensitivity:",
    continuous_update=False,
    readout_format='.2f',
    layout=widgets.Layout(width='300px')
)

sigma_slider = widgets.IntSlider(
    value=20, min=5, max=100, step=1,
    description="Snap Range (px):",
    continuous_update=False,
    layout=widgets.Layout(width='300px')
)

btn_start = widgets.Button(description="Start Inspection", button_style='primary')
btn_keep = widgets.Button(description="✅ KEEP BEAD", button_style='success', layout=widgets.Layout(width='150px'))
btn_reject = widgets.Button(description="❌ REJECT", button_style='danger', layout=widgets.Layout(width='150px'))
btn_undo = widgets.Button(description="Undo", button_style='warning', disabled=True)
btn_finish = widgets.Button(description="Finish & Move Rejected", button_style='info', disabled=True)

progress_label = widgets.Label(value="Waiting to start...")
plot_output = widgets.Output()

# --- Logic ---

def refine_centroid(vol, click_yx, search_radius):
    y_click, x_click = click_yx
    h, w = vol.shape[1], vol.shape[2]
    
    y1 = max(0, int(y_click - search_radius))
    y2 = min(h, int(y_click + search_radius))
    x1 = max(0, int(x_click - search_radius))
    x2 = min(w, int(x_click + search_radius))
    
    if y2 <= y1 or x2 <= x1: return click_yx
    
    mip_window = np.max(vol[:, y1:y2, x1:x2], axis=0)
    bg = np.min(mip_window)
    weights = mip_window - bg
    
    if np.sum(weights) == 0: return click_yx 
    
    cy_local, cx_local = ndimage.center_of_mass(weights)
    return (y1 + cy_local, x1 + cx_local)

def get_local_mask(vol, center, radius, sensitivity):
    # 1. Create Geometric Mask (Cylinder)
    cy, cx = center
    h, w = vol.shape[1], vol.shape[2]
    Y, X = np.ogrid[:h, :w]
    dist_sq = (Y - cy)**2 + (X - cx)**2
    mask_geo = np.broadcast_to((dist_sq <= radius**2)[None, :, :], vol.shape)
    
    # 2. Extract ONLY pixels inside the cylinder for threshold calc
    # This prevents bright neighbors from skewing the sensitivity
    valid_pixels = vol[mask_geo]
    
    if len(valid_pixels) == 0:
        return mask_geo # Should not happen if radius > 0
    
    # Calculate Otsu on valid pixels only
    try:
        thresh = threshold_otsu(valid_pixels)
    except:
        thresh = np.percentile(valid_pixels, 90)
    
    # Apply Sensitivity
    s = max(sensitivity, 0.01)
    eff_thresh = thresh / s
    
    # 3. Create Intensity Mask (Whole Volume)
    # We apply the threshold to the whole volume first
    smooth = ndimage.gaussian_filter(vol, sigma=1.0)
    mask_thresh = smooth > eff_thresh
    mask_thresh = ndimage.binary_dilation(mask_thresh, iterations=1)
    
    # 4. Intersection
    return mask_geo & mask_thresh

def calculate_bg(vol, mask):
    bg_pixels = vol[~mask]
    if len(bg_pixels) == 0: return np.percentile(vol, 5)
    return np.median(bg_pixels)

def update_mask_and_plots():
    if state["current_vol"] is None or state["centroid"] is None: return
    
    state["current_mask"] = get_local_mask(
        state["current_vol"], 
        state["centroid"], 
        radius_slider.value, 
        sens_slider.value
    )
    draw_plots()

def on_canvas_click(event):
    if event.inaxes != state["axes"]["xy"]: return
    if state["current_vol"] is None: return
    
    refined_y, refined_x = refine_centroid(
        state["current_vol"], 
        (event.ydata, event.xdata), 
        sigma_slider.value
    )
    state["centroid"] = (refined_y, refined_x)
    update_mask_and_plots()

def load_bead(idx):
    if idx >= len(state["files"]):
        with plot_output:
            clear_output()
            print("🎉 Inspection Complete!")
            print(f"Kept: {len(state['kept'])}")
            print(f"Rejected: {len(state['rejected'])}")
        btn_keep.disabled = True
        btn_reject.disabled = True
        btn_finish.disabled = False
        return

    f = state["files"][idx]
    vol = tifffile.imread(f).astype(np.float32)
    if vol.ndim > 3: vol = np.squeeze(vol)
    state["current_vol"] = vol
    
    if state["fig"] is None:
        with plot_output:
            clear_output(wait=True)
            fig = plt.figure(figsize=(10, 8))
            gs = fig.add_gridspec(2, 2)
            state["fig"] = fig
            state["axes"] = {
                "xz": fig.add_subplot(gs[0, 0]),
                "xy": fig.add_subplot(gs[0, 1]),
                "yz": fig.add_subplot(gs[1, 0]),
                "prof": fig.add_subplot(gs[1, 1])
            }
            state["cid"] = fig.canvas.mpl_connect('button_press_event', on_canvas_click)
            plt.show()

    cy, cx = vol.shape[1]/2, vol.shape[2]/2
    state["centroid"] = refine_centroid(vol, (cy, cx), sigma_slider.value * 2)
    
    progress_label.value = f"File {idx+1}/{len(state['files'])}: {f.name}"
    update_mask_and_plots()

def draw_plots():
    vol = state["current_vol"]
    mask = state["current_mask"]
    cy, cx = state["centroid"]
    bg_val = calculate_bg(vol, mask)
    cz = np.argmax(vol[:, int(cy), int(cx)])
    r = radius_slider.value
    
    # --- XZ View ---
    ax = state["axes"]["xz"]
    ax.clear()
    ax.imshow(vol[:, int(cy), :], cmap='gray', origin='upper', aspect='auto')
    
    # Geometric Limits
    ax.axvline(cx - r, color='cyan', linestyle='--', alpha=0.5)
    ax.axvline(cx + r, color='cyan', linestyle='--', alpha=0.5)
    
    # Mask
    if np.any(mask[:, int(cy), :]):
        ax.contour(mask[:, int(cy), :], levels=[0.5], colors='red', linewidths=1)
        
    ax.set_title("XZ View (Side)")
    ax.set_ylabel("Z")

    # --- XY View (Top) ---
    ax = state["axes"]["xy"]
    ax.clear()
    ax.imshow(vol[cz, :, :], cmap='gray', origin='upper')
    
    circ = Circle((cx, cy), r, edgecolor='cyan', facecolor='none', linewidth=1.5, linestyle='--')
    ax.add_patch(circ)
    ax.plot(cx, cy, 'c+') 
    
    if np.any(mask[cz, :, :]):
        ax.contour(mask[cz, :, :], levels=[0.5], colors='red', linewidths=1)

    ax.set_title("XY View (Top) - Click to Move")

    # --- YZ View ---
    ax = state["axes"]["yz"]
    ax.clear()
    ax.imshow(vol[:, :, int(cx)].T, cmap='gray', origin='upper', aspect='auto')
    
    if np.any(mask[:, :, int(cx)]):
        ax.contour(mask[:, :, int(cx)].T, levels=[0.5], colors='red', linewidths=1)
        
    ax.set_title("YZ View (Front)")
    ax.set_ylabel("Y")

    # --- Profile ---
    ax = state["axes"]["prof"]
    ax.clear()
    prof = vol[:, int(cy), int(cx)]
    ax.plot(prof, label='Z-Signal', color='blue')
    ax.axhline(bg_val, color='red', linestyle='--', label=f'BG: {bg_val:.1f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    state["fig"].canvas.draw_idle()

# --- Actions ---
def on_start_click(b):
    if not folder_chooser.selected: return
    p = Path(folder_chooser.selected)
    all_files = sorted(list(p.glob("*.tif")) + list(p.glob("*.tiff")))
    state["files"] = [f for f in all_files if "Master" not in f.name and "Avg" not in f.name and "_mask" not in f.name]
    if not state["files"]: return
    
    state["current_idx"] = 0
    state["kept"] = []
    state["rejected"] = []
    btn_start.disabled = True
    btn_keep.disabled = False
    btn_reject.disabled = False
    load_bead(0)

def on_keep(b):
    current_file = state["files"][state["current_idx"]]
    mask_name = current_file.parent / (current_file.stem + "_mask.tif")
    mask_uint8 = (state["current_mask"] * 255).astype(np.uint8)
    tifffile.imwrite(mask_name, mask_uint8)
    state["kept"].append(current_file)
    state["current_idx"] += 1
    btn_undo.disabled = False
    load_bead(state["current_idx"])

def on_reject(b):
    state["rejected"].append(state["files"][state["current_idx"]])
    state["current_idx"] += 1
    btn_undo.disabled = False
    load_bead(state["current_idx"])

def on_undo(b):
    if state["current_idx"] > 0:
        state["current_idx"] -= 1
        f = state["files"][state["current_idx"]]
        if f in state["kept"]: 
            state["kept"].remove(f)
            mask_name = f.parent / (f.stem + "_mask.tif")
            if mask_name.exists(): mask_name.unlink()
        if f in state["rejected"]: state["rejected"].remove(f)
        load_bead(state["current_idx"])

def on_finish(b):
    if not state["rejected"]: return
    rej_dir = Path(folder_chooser.selected) / "rejected_beads"
    rej_dir.mkdir(exist_ok=True)
    for f in state["rejected"]:
        try: shutil.move(str(f), str(rej_dir / f.name))
        except: pass
    print(f"Moved {len(state['rejected'])} files to {rej_dir.name}")

# --- Wiring ---
btn_start.on_click(on_start_click)
radius_slider.observe(lambda c: update_mask_and_plots(), names='value')
sens_slider.observe(lambda c: update_mask_and_plots(), names='value')
sigma_slider.observe(lambda c: update_mask_and_plots(), names='value')
btn_keep.on_click(on_keep)
btn_reject.on_click(on_reject)
btn_undo.on_click(on_undo)
btn_finish.on_click(on_finish)

display(widgets.VBox([
    folder_chooser,
    btn_start,
    widgets.HBox([plot_output]),
    progress_label,
    widgets.HBox([radius_slider, sens_slider, sigma_slider]),
    widgets.HBox([btn_keep, btn_reject, btn_undo]),
    widgets.HTML("<hr>"),
    btn_finish
]))

In [ ]:
# --- Averager v4 (Strict "Cookie Cutter" Mode) ---
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
import tifffile
import numpy as np
from pathlib import Path
from scipy import ndimage, signal, optimize
from skimage.filters import threshold_otsu

print("Select Folder containing your KEPT Bead ROIs:")
folder_chooser = FileChooser(
    path="/mmfs2/scratch/SDSMT.LOCAL/bscott/DataUpload",
    title="<b>Select Folder:</b>",
    show_only_dirs=True
)

# New Control: Strict Masking
strict_mask_check = widgets.Checkbox(value=True, description="Hard Crop (Zero outside mask)")
taper_slider = widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05, description="Edge Taper:")

btn_run = widgets.Button(description="Generate Master PSF", button_style='success', layout=widgets.Layout(width='200px'))
out_log = widgets.Output(layout={'border': '1px solid #ccc', 'height': '200px', 'overflow_y': 'scroll'})
out_plot = widgets.Output()

def fit_gaussian_1d(x, y):
    def gauss(x, a, x0, sigma, offset):
        return a * np.exp(-(x - x0)**2 / (2 * sigma**2)) + offset
    try:
        p0 = [np.max(y), len(y)/2, len(y)/4, 0]
        popt, _ = optimize.curve_fit(gauss, x, y, p0=p0)
        return 2.355 * abs(popt[2]) 
    except:
        return 0.0

def get_fallback_mask(vol):
    smooth = ndimage.gaussian_filter(vol, sigma=1.0)
    try: thresh = threshold_otsu(smooth)
    except: thresh = np.percentile(smooth, 90)
    mask = smooth > thresh
    return ndimage.binary_dilation(mask, iterations=2)

def taper_vol(vol, fraction=0.1):
    d, h, w = vol.shape
    win_z = signal.windows.tukey(d, alpha=fraction)
    win_y = signal.windows.tukey(h, alpha=fraction)
    win_x = signal.windows.tukey(w, alpha=fraction)
    return vol * (win_z[:,None,None] * win_y[None,:,None] * win_x[None,None,:])

def run_average(b):
    with out_log:
        clear_output()
        if not folder_chooser.selected: return
        p = Path(folder_chooser.selected)
        
        all_tifs = sorted(list(p.glob("*.tif")) + list(p.glob("*.tiff")))
        bead_files = [f for f in all_tifs if "Master" not in f.name and "Avg" not in f.name and "_mask" not in f.name]
        
        if not bead_files:
            print("❌ No beads found.")
            return
            
        print(f"--- Averaging {len(bead_files)} Beads ---")
        
        # Determine Canvas
        shapes = [tifffile.imread(f).shape for f in bead_files]
        mz = max((s[0] if len(s)==3 else 1) for s in shapes)
        my = max((s[1] if len(s)==3 else s[0]) for s in shapes)
        mx = max((s[2] if len(s)==3 else s[1]) for s in shapes)
        mz += 1 if mz%2==0 else 0
        my += 1 if my%2==0 else 0
        mx += 1 if mx%2==0 else 0
        target_shape = (mz, my, mx)
        center_target = np.array(target_shape) / 2.0
        
        aligned_sum = np.zeros(target_shape, dtype=np.float32)
        used_count = 0
        
        for f in bead_files:
            vol = tifffile.imread(f).astype(np.float32)
            if vol.ndim > 3: vol = np.squeeze(vol)
            
            # 1. LOAD MASK
            mask_path = f.parent / (f.stem + "_mask.tif")
            if mask_path.exists():
                mask = tifffile.imread(mask_path) > 0
                mask_source = "USER"
            else:
                mask = get_fallback_mask(vol)
                mask_source = "AUTO"
            
            # 2. Subtract BG
            bg_pixels = vol[~mask]
            bg_val = np.median(bg_pixels) if len(bg_pixels) > 0 else 0
            vol_clean = vol - bg_val
            vol_clean[vol_clean < 0] = 0
            
            # 3. STRICT MASKING (The Cookie Cutter)
            # If enabled, force everything outside the mask to exactly 0
            if strict_mask_check.value:
                vol_clean[~mask] = 0.0
            
            # 4. Center of Mass (Weighted by Clean Signal)
            if np.sum(vol_clean) == 0:
                print(f"⚠️ Skipped {f.name} (Empty after cleaning)")
                continue
                
            com = np.array(ndimage.center_of_mass(vol_clean))
            
            # 5. Embed & Shift
            canvas = np.zeros(target_shape, dtype=np.float32)
            z_off = (mz - vol.shape[0]) // 2
            y_off = (my - vol.shape[1]) // 2
            x_off = (mx - vol.shape[2]) // 2
            canvas[z_off:z_off+vol.shape[0], y_off:y_off+vol.shape[1], x_off:x_off+vol.shape[2]] = vol_clean
            
            com_canvas = com + np.array([z_off, y_off, x_off])
            shift_vec = center_target - com_canvas
            
            aligned = ndimage.shift(canvas, shift_vec, order=3, prefilter=True)
            aligned[aligned < 0] = 0
            
            # Normalize Peak
            if np.max(aligned) > 0: aligned /= np.max(aligned)
            aligned_sum += aligned
            used_count += 1
            
            print(f"[{mask_source}] Merged: {f.name}")

        if used_count == 0: return

        # Final Average
        master_psf = aligned_sum / used_count
        
        # Taper
        if taper_slider.value > 0:
            master_psf = taper_vol(master_psf, fraction=taper_slider.value)
            
        master_psf /= np.max(master_psf)
        save_path = p / "Master_PSF_Final_Strict.tif"
        tifffile.imwrite(save_path, master_psf.astype(np.float32))
        print(f"\n✅ SAVED: {save_path.name}")
        
        # --- Visualization ---
        with out_plot:
            clear_output()
            
            cz, cy, cx = list(map(int, center_target))
            prof_z = master_psf[:, cy, cx]
            prof_x = master_psf[cz, cy, :]
            
            px_xy, px_z = 0.136, 0.100 
            fz = fit_gaussian_1d(np.arange(len(prof_z)), prof_z) * px_z
            fx = fit_gaussian_1d(np.arange(len(prof_x)), prof_x) * px_xy
            
            fig = plt.figure(figsize=(10, 6))
            gs = fig.add_gridspec(2, 3)
            
            # Log Scale helps see if background is truly gone
            ax0 = fig.add_subplot(gs[0, 0])
            ax0.imshow(np.log1p(master_psf[:, cy, :]), cmap='magma', aspect='auto')
            ax0.set_title(f"XZ View (Log Scale)\nFWHM Z: {fz:.3f} µm")
            
            ax1 = fig.add_subplot(gs[0, 1])
            ax1.imshow(np.log1p(master_psf[cz, :, :]), cmap='magma')
            ax1.set_title(f"XY View (Log Scale)\nFWHM X: {fx:.3f} µm")
            
            ax3 = fig.add_subplot(gs[1, :])
            ax3.plot(prof_z, label='Z Profile', color='blue')
            ax3.plot(prof_x, label='X Profile', color='orange', linestyle='--')
            ax3.set_title("Intensity Profiles")
            ax3.legend()
            ax3.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()

btn_run.on_click(run_average)
display(widgets.VBox([folder_chooser, widgets.HBox([strict_mask_check, taper_slider]), btn_run, out_log, out_plot]))

In [ ]:
# --- Interactive Master PSF Slicer ---
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
import tifffile
import numpy as np
from pathlib import Path

print("Select your Master PSF File (e.g., Master_PSF_Final_Strict.tif):")
file_chooser = FileChooser(
    path="/mmfs2/scratch/SDSMT.LOCAL/bscott/DataUpload",
    title="<b>Select Master PSF:</b>",
    filter_pattern=['*.tif', '*.tiff']
)

# --- State ---
state = {
    "vol": None,
    "view": "XY", # XY, XZ, YZ
    "fig": None,
    "ax": None,
    "img_obj": None
}

# --- Controls ---
slice_slider = widgets.IntSlider(description="Slice:", continuous_update=True, layout=widgets.Layout(width='400px'))
contrast_slider = widgets.FloatRangeSlider(
    value=[0, 1], min=0, max=1, step=0.01, 
    description="Contrast:", 
    layout=widgets.Layout(width='400px')
)
gamma_slider = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description="Gamma:")

btn_xy = widgets.Button(description="XY (Top)", button_style='info')
btn_xz = widgets.Button(description="XZ (Side)", button_style='info')
btn_yz = widgets.Button(description="YZ (Front)", button_style='info')

plot_output = widgets.Output()

def load_file(b):
    if not file_chooser.selected: return
    f = Path(file_chooser.selected)
    print(f"Loading {f.name}...")
    
    vol = tifffile.imread(f).astype(np.float32)
    if vol.ndim > 3: vol = np.squeeze(vol)
    
    # Normalize to 0-1 for display
    vol -= np.min(vol)
    m = np.max(vol)
    if m > 0: vol /= m
    
    state["vol"] = vol
    switch_view("XY")

def switch_view(view_mode):
    state["view"] = view_mode
    vol = state["vol"]
    if vol is None: return
    
    # Reset Slider Range based on view depth
    if view_mode == "XY":
        # Z-stack is the slider
        max_slice = vol.shape[0] - 1
        val = max_slice // 2
    elif view_mode == "XZ":
        # Y-stack is the slider
        max_slice = vol.shape[1] - 1
        val = max_slice // 2
    else: # YZ
        # X-stack is the slider
        max_slice = vol.shape[2] - 1
        val = max_slice // 2
        
    slice_slider.max = max_slice
    slice_slider.value = val
    
    # Update buttons look
    btn_xy.button_style = 'primary' if view_mode=='XY' else 'info'
    btn_xz.button_style = 'primary' if view_mode=='XZ' else 'info'
    btn_yz.button_style = 'primary' if view_mode=='YZ' else 'info'
    
    # Init Plot
    with plot_output:
        clear_output(wait=True)
        if state["fig"] is None:
            fig, ax = plt.subplots(figsize=(7, 7))
            state["fig"] = fig
            state["ax"] = ax
        else:
            state["ax"].clear()
            
        update_slice(None)
        plt.show()

def get_current_slice():
    vol = state["vol"]
    idx = slice_slider.value
    mode = state["view"]
    
    if mode == "XY":
        return vol[idx, :, :]
    elif mode == "XZ":
        return vol[:, idx, :]
    else: # YZ
        return vol[:, :, idx].T # Transpose to keep Z vertical

def update_slice(change):
    if state["vol"] is None or state["ax"] is None: return
    
    sl = get_current_slice()
    
    # Apply Gamma
    g = gamma_slider.value
    if g != 1.0:
        sl = sl ** g
    
    vmin, vmax = contrast_slider.value
    
    if state["img_obj"] is None or state["img_obj"].axes != state["ax"]:
        state["img_obj"] = state["ax"].imshow(sl, cmap='magma', vmin=vmin, vmax=vmax, origin='upper', aspect='auto')
        state["ax"].set_title(f"{state['view']} View - Slice {slice_slider.value}")
    else:
        state["img_obj"].set_data(sl)
        state["img_obj"].set_clim(vmin, vmax)
        state["ax"].set_title(f"{state['view']} View - Slice {slice_slider.value}")
        
    state["fig"].canvas.draw_idle()

# --- Wiring ---
file_chooser.register_callback(load_file)
slice_slider.observe(update_slice, names='value')
contrast_slider.observe(update_slice, names='value')
gamma_slider.observe(update_slice, names='value')

btn_xy.on_click(lambda b: switch_view("XY"))
btn_xz.on_click(lambda b: switch_view("XZ"))
btn_yz.on_click(lambda b: switch_view("YZ"))

display(widgets.VBox([
    file_chooser,
    widgets.HBox([btn_xy, btn_xz, btn_yz]),
    slice_slider,
    widgets.HBox([contrast_slider, gamma_slider]),
    plot_output
]))

In [ ]:
# --- Deconvolution Results Viewer (MIPs + Slicing) ---
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
import tifffile
import numpy as np
from pathlib import Path

print("Select your Deconvolved File:")
file_chooser = FileChooser(
    path="/mmfs2/scratch/SDSMT.LOCAL/bscott/DataUpload",
    title="<b>Select Image (.tif):</b>",
    filter_pattern=['*.tif', '*.tiff']
)

# --- State ---
state = {
    "vol": None,
    "mips": {},
    "shape": (0,0,0), # Z, Y, X
    "fig": None,
    "axes": {},
    "mode": "MIP" # or "SLICE"
}

# --- Controls ---
mode_toggle = widgets.ToggleButtons(
    options=['MIP (Max Projection)', 'Slice Explorer'],
    description='View Mode:',
    button_style='info'
)

# Sliders for Slicing
z_slider = widgets.IntSlider(description="Z Slice:", layout=widgets.Layout(width='300px'))
y_slider = widgets.IntSlider(description="Y Slice:", layout=widgets.Layout(width='300px'))
x_slider = widgets.IntSlider(description="X Slice:", layout=widgets.Layout(width='300px'))

# Contrast
contrast_slider = widgets.FloatRangeSlider(
    value=[0, 1], min=0, max=1, step=0.01,
    description="Contrast:", layout=widgets.Layout(width='400px')
)

plot_output = widgets.Output()

def load_data(b):
    if not file_chooser.selected: return
    f = Path(file_chooser.selected)
    print(f"Loading {f.name}...")
    
    # Load Volume
    vol = tifffile.imread(f).astype(np.float32)
    if vol.ndim > 3: vol = np.squeeze(vol)
    
    # Normalize for display (0-1)
    vol -= np.min(vol)
    m = np.max(vol)
    if m > 0: vol /= m
    
    state["vol"] = vol
    state["shape"] = vol.shape
    
    # Pre-calculate MIPs
    print("Calculating MIPs...")
    state["mips"] = {
        "xy": np.max(vol, axis=0),
        "xz": np.max(vol, axis=1),
        "yz": np.max(vol, axis=2) # Will transpose later
    }
    
    # Setup Sliders
    z_slider.max = vol.shape[0] - 1
    y_slider.max = vol.shape[1] - 1
    x_slider.max = vol.shape[2] - 1
    
    z_slider.value = vol.shape[0] // 2
    y_slider.value = vol.shape[1] // 2
    x_slider.value = vol.shape[2] // 2
    
    # Setup Plot
    init_plot()
    update_display()
    print("✅ Loaded.")

def init_plot():
    with plot_output:
        clear_output(wait=True)
        fig = plt.figure(figsize=(10, 6))
        gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1])
        
        # Layout:
        # Top-Left: XY (Top View)
        # Bottom-Left: XZ (Side View)
        # Top-Right: YZ (Front View)
        # Bottom-Right: Histogram or Info?
        
        ax_xy = fig.add_subplot(gs[0, 0])
        ax_xz = fig.add_subplot(gs[1, 0])
        ax_yz = fig.add_subplot(gs[0, 1])
        
        state["fig"] = fig
        state["axes"] = {"xy": ax_xy, "xz": ax_xz, "yz": ax_yz}
        plt.show()

def update_display(change=None):
    if state["vol"] is None or state["fig"] is None: return
    
    vmin, vmax = contrast_slider.value
    cmap = 'magma'
    
    # Prepare Images
    if mode_toggle.value == 'MIP (Max Projection)':
        img_xy = state["mips"]["xy"]
        img_xz = state["mips"]["xz"]
        img_yz = state["mips"]["yz"].T # Transpose YZ to keep Z up
        title_prefix = "MIP"
        
        # Disable slice sliders in MIP mode
        z_slider.disabled = True
        y_slider.disabled = True
        x_slider.disabled = True
        
    else: # SLICE MODE
        cz, cy, cx = z_slider.value, y_slider.value, x_slider.value
        img_xy = state["vol"][cz, :, :]
        img_xz = state["vol"][:, cy, :]
        img_yz = state["vol"][:, :, cx].T
        title_prefix = "Slice"
        
        # Enable sliders
        z_slider.disabled = False
        y_slider.disabled = False
        x_slider.disabled = False
    
    # Draw XY
    ax = state["axes"]["xy"]
    ax.clear()
    ax.imshow(img_xy, cmap=cmap, vmin=vmin, vmax=vmax, origin='upper')
    ax.set_title(f"{title_prefix} XY (Top)")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    
    # Draw XZ
    ax = state["axes"]["xz"]
    ax.clear()
    ax.imshow(img_xz, cmap=cmap, vmin=vmin, vmax=vmax, origin='upper', aspect='auto')
    ax.set_title(f"{title_prefix} XZ (Side)")
    ax.set_xlabel("X")
    ax.set_ylabel("Z")
    
    # Draw YZ
    ax = state["axes"]["yz"]
    ax.clear()
    ax.imshow(img_yz, cmap=cmap, vmin=vmin, vmax=vmax, origin='upper', aspect='auto')
    ax.set_title(f"{title_prefix} YZ (Front)")
    ax.set_xlabel("Z")
    ax.set_ylabel("Y")
    
    # Add Crosshairs if in Slice Mode
    if mode_toggle.value == 'Slice Explorer':
        cz, cy, cx = z_slider.value, y_slider.value, x_slider.value
        
        # XY View: Show X and Y lines
        state["axes"]["xy"].axvline(cx, color='cyan', alpha=0.5)
        state["axes"]["xy"].axhline(cy, color='cyan', alpha=0.5)
        
        # XZ View: Show X and Z lines
        state["axes"]["xz"].axvline(cx, color='cyan', alpha=0.5)
        state["axes"]["xz"].axhline(cz, color='cyan', alpha=0.5)
        
        # YZ View: Show Z and Y lines
        state["axes"]["yz"].axvline(cz, color='cyan', alpha=0.5) # Z is X-axis here
        state["axes"]["yz"].axhline(cy, color='cyan', alpha=0.5)

    state["fig"].canvas.draw_idle()

# --- Wiring ---
file_chooser.register_callback(load_data)
mode_toggle.observe(update_display, names='value')
z_slider.observe(update_display, names='value')
y_slider.observe(update_display, names='value')
x_slider.observe(update_display, names='value')
contrast_slider.observe(update_display, names='value')

display(widgets.VBox([
    file_chooser,
    widgets.HBox([mode_toggle, contrast_slider]),
    widgets.HBox([z_slider, y_slider, x_slider]),
    plot_output
]))